In [154]:
import pandas as pd
import numpy as np
import faiss

# 1. EDA

In [155]:
df = pd.read_csv('dataset/movielens_100k.csv')

df.head(5)

,movie_id,title,year,directors,actors,genres
0,1,toy story,1995,John Lasseter,Tom Hanks Tim Allen Don Rickles Jim Varney Wal...,Animation Adventure Comedy Family Fantasy
1,2,goldeneye,1995,Martin Campbell,Pierce Brosnan Sean Bean Izabella Scorupco Fam...,Action Adventure Thriller
2,3,four rooms,1995,Allison Anders Alexandre Rockwell Robert Rodri...,Sammi Davis Amanda De Cadenet Valeria Golino M...,Comedy
3,4,get shorty,1995,Barry Sonnenfeld,John Travolta Gene Hackman Rene Russo Danny De...,Comedy Crime Thriller
4,5,copycat,1995,Jon Amiel,Sigourney Weaver Holly Hunter Dermot Mulroney ...,Drama Mystery Thriller


In [156]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1681 entries, 0 to 1680
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   movie_id   1681 non-null   int64
 1   title      1681 non-null   str  
 2   year       1681 non-null   int64
 3   directors  1555 non-null   str  
 4   actors     1555 non-null   str  
 5   genres     1561 non-null   str  
dtypes: int64(2), str(4)
memory usage: 78.9 KB


In [157]:
for i in df.columns:
    if i == 'genres':# or i == 'actors':
        print(df[i].unique())

<StringArray>
['Animation Adventure Comedy Family Fantasy',
                 'Action Adventure Thriller',
                                    'Comedy',
                     'Comedy Crime Thriller',
                    'Drama Mystery Thriller',
                                         nan,
                   'Mystery Sci-Fi Thriller',
                       'Comedy Drama Family',
                               'Crime Drama',
                          'Drama Sci-Fi War',
 ...
                    'Comedy Fantasy Mystery',
                           'Music Talk-Show',
              'Comedy Crime Horror Thriller',
        'Comedy Drama Fantasy Romance Sport',
               'Documentary Biography Music',
              'Comedy Drama Fantasy Mystery',
            'Documentary Comedy Crime Drama',
              'Comedy Drama Romance Fantasy',
            'Adventure Comedy Crime Mystery',
                        'Documentary Comedy']
Length: 460, dtype: str


In [158]:
# Removendo dados com diretor ou genero sem informação
df = df.dropna(subset = ['genres', 'directors'])
df = df.reset_index(drop=True)

In [159]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1553 entries, 0 to 1552
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   movie_id   1553 non-null   int64
 1   title      1553 non-null   str  
 2   year       1553 non-null   int64
 3   directors  1553 non-null   str  
 4   actors     1546 non-null   str  
 5   genres     1553 non-null   str  
dtypes: int64(2), str(4)
memory usage: 72.9 KB


# 2. Seleção dos dados e dummyzação

In [160]:
# Selecionando a coluna genero
sentencas_gen = df[['genres']]
sentencas_gen.head(5)

,genres
0,Animation Adventure Comedy Family Fantasy
1,Action Adventure Thriller
2,Comedy
3,Comedy Crime Thriller
4,Drama Mystery Thriller


In [161]:
# Aplicando one hot econding para garantir que os textos se tornem colunas binárias
sentencas_gen_bin = sentencas_gen['genres'].str.get_dummies(' ')

In [162]:
# Conversão para vetores, colocando eles em formato especifico na memoria FLOAT 32
vet_gen = sentencas_gen_bin.to_numpy().astype('float32')

In [163]:
print(sentencas_gen_bin.columns)

Index(['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'History',
       'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Romance', 'Sci-Fi',
       'Short', 'Sport', 'Talk-Show', 'Thriller', 'War', 'Western'],
      dtype='str')


# 3. Aplicação de Faiss  

## 3.1. Flat L2 Index  
Extremamente preciso, mas lento

In [164]:
# Definindo a dimensão do vetor
d = vet_gen.shape[1]
d

25

In [165]:
# Inicializando o Flat L2
index_l2 = faiss.IndexFlatL2(d)

O Index L2, diferente dos demais mais utilizados, não precisa de treino.  
Treino é o processo de identificar os limites de onde será comparado pra obter os dados. **Voroni Cells**

In [166]:
index_l2.is_trained

True

In [167]:
# Adicionando os objetos no vetor criado (index)
index_l2.add(vet_gen)

In [168]:
vet_gen[10:11]

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
        0., 0., 0., 0., 0., 0., 1., 0., 0.]], dtype=float32)

In [169]:
# Escolha do genero do filme
vector_film = df[df['genres'] == 'Thriller']
print(vector_film)

vector_film = vector_film.index[0]

      movie_id                  title  year         directors  \
939        983     rich mans wife the  1996  Amy Holden Jones   
1311      1399  stranger in the house  1997    Rodney Gibbons   
1337      1431           legal deceit  1997     Monika Harris   
1441      1549              dream man  1995     René Bonnière   

                                                 actors    genres  
939   Halle Berry Peter Greene Clive Owen Frankie Fa...  Thriller  
1311  Michele Greene Bruce Dinsmore Steve Railsback ...  Thriller  
1337  Lela Rochon Jeffrey Dean Morgan Phil Morris Ch...  Thriller  
1441  Patsy Kensit Bruce Greenwood Andrew McCarthy D...  Thriller  


In [170]:
# Iniciando o processo de busca dos 3 mais proximos (semelhantes)
k = 10

# O que queremos encontrar 
query = vet_gen[[vector_film]]# o resultado será generos semelhantes ao existentes nesse vetor

In [171]:
%%time

# Busca
dim, ind = index_l2.search(query, k)

# Resultado de filmes com generos identicos
print(df.iloc[ind[0]][['title', 'genres']])

                       title           genres
939       rich mans wife the         Thriller
1311   stranger in the house         Thriller
1337            legal deceit         Thriller
1441               dream man         Thriller
41                disclosure   Drama Thriller
139                 die hard  Action Thriller
190   nikita la femme nikita  Action Thriller
209                cape fear   Crime Thriller
217               die hard 2  Action Thriller
224              under siege  Action Thriller
CPU times: user 10.4 ms, sys: 1.99 ms, total: 12.4 ms
Wall time: 19.6 ms


## 3.2. Flat IVF Index  
Alta precisão com um rapida velocidade

In [172]:
# Definir o quantizador, responsavel por separar os vetor em grupos (clusterização)
quant = faiss.IndexFlatL2(d)

In [173]:
# Definir o numero de grupos criados, que precisa ser um numero de divisao exata
n = sentencas_gen.shape[0]
n = np.sqrt(n)
n

np.float64(39.408120990476064)

In [174]:
nlist = int(n)
index_ivf = faiss.IndexIVFFlat(quant, d, nlist)

Como esse metodo trabalha somente com alguns dados e não todos, ele precisa ser treinado para avaliar

In [175]:
index_ivf.train(vet_gen)

In [176]:
# Adicionando os objetos no vetor criado
index_ivf.add(vet_gen)

In [177]:
%%time
# Busca
dim, ind = index_ivf.search(query, k)

# Resultado de filmes com generos identicos
print(df.iloc[ind[0]][['title', 'genres']])

                         title          genres
939         rich mans wife the        Thriller
1311     stranger in the house        Thriller
1337              legal deceit        Thriller
1441                 dream man        Thriller
41                  disclosure  Drama Thriller
317                        187  Drama Thriller
443    cry the beloved country  Drama Thriller
444         crossing guard the  Drama Thriller
537          white mans burden  Drama Thriller
633   manchurian candidate the  Drama Thriller
CPU times: user 12.4 ms, sys: 1.75 ms, total: 14.2 ms
Wall time: 20.2 ms


In [178]:
%%time
# Apresentando L2 novamente so para comparar os resultados 
# Busca
dim, ind = index_l2.search(query, k)

# Resultado de filmes com generos identicos
print(df.iloc[ind[0]][['title', 'genres']])

                       title           genres
939       rich mans wife the         Thriller
1311   stranger in the house         Thriller
1337            legal deceit         Thriller
1441               dream man         Thriller
41                disclosure   Drama Thriller
139                 die hard  Action Thriller
190   nikita la femme nikita  Action Thriller
209                cape fear   Crime Thriller
217               die hard 2  Action Thriller
224              under siege  Action Thriller
CPU times: user 9.57 ms, sys: 1.76 ms, total: 11.3 ms
Wall time: 17.8 ms


Apesar dos 4 primeiros indices representarem o mesmo filme, os demais se encontram em posicṍes diferentes ou até mesmo são diferentes. Isso se justifica dado a maior precisão de L2, que em um cenário real para big data também deveria demorar mais se comparado. Com isso, é coerente afirmar que o IVF é superior somente diante de dados de produção, podendo ser facilmente substituido por L2 nesses dados.